# Device-Level Analysis: Stack Architecture and Performance

## 1. Abstract
This notebook transitions from material properties to full device engineering. We analyze the impact of charge transport layers (ETL/HTL) and architecture (n-i-p vs p-i-n) on the experimental outcomes. The objective is to identify which interface combinations maximize both efficiency and durability by mapping complex stack sequences to physical performance metrics.

## 2. Theoretical Framework: The Physics-to-ML Mapping

### 2.1. Arrhenius Normalization ($T_{acc}$)
The degradation of perovskite solar cells is often thermally activated, following the Arrhenius relationship:
$$
k = A e^{-\frac{E_a}{R T}}
$$
Where $k$ is the degradation rate, $E_a$ is the activation energy, $R$ is the gas constant, and $T$ is the absolute temperature. To compare devices tested at different temperatures ($T_{exp}$), we normalize their lifetimes ($T_{80}$) to a baseline temperature ($T_{ref} = 298.15\text{ K}$):
$$
T_{std} = T_{exp} \cdot \exp\left[ \frac{E_a}{R} \left( \frac{1}{T_{ref}} - \frac{1}{T_{exp}} \right) \right]
$$

*   **Physics Role:** It standardizes lifetimes from varied test conditions (65°C, 85°C) to a common baseline. It assumes an average $E_a$ (typically $\approx 0.6\text{ eV}$) for the dominant degradation mechanism (e.g., ion migration).
*   **ML Connection:** This provides **Target Regularization**. By normalizing the target variable, the CatBoost model avoids learning 'spurious correlations' between high-stress lab conditions and device quality, allowing it to isolate the intrinsic stability contribution of the chemical composition and stack architecture.

### 2.2. Acceleration Humidity ($RH$)
Moisture induces hydration of organic cations (like methylammonium, $MA^+$), leading to lattice dissolution:
$$
CH_3NH_3PbI_3 + H_2O \leftrightarrow CH_3NH_3PbI_3 \cdot H_2O \rightarrow PbI_2 + CH_3NH_3I(aq)
$$
*   **Physics Role:** Humidity acts as a chemical catalyst for degradation. High $RH$ levels significantly reduce the energy barrier for lattice collapse.
*   **ML Connection:** This acts as a **Non-Linear Interaction Term**. The model captures how moisture sensitivity is modulated by the **Octahedral Factor** ($u$) and **Goldschmidt Tolerance Factor** ($t$). For example, compositions with low $t$ (strained lattices) show a much steeper decay in predicted $T_{80}$ as $RH$ increases compared to more stable 'triple cation' systems.

### 2.3. HTL/ETL Sequences: Interface Physics
The power conversion efficiency (PCE, $\eta$) is defined by the product of short-circuit current ($J_{sc}$), open-circuit voltage ($V_{oc}$), and fill factor ($FF$), divided by the incident power ($P_{in}$):
$$
\eta = \frac{J_{sc} \cdot V_{oc} \cdot FF}{P_{in}}
$$
The transport layers (HTL/ETL) minimize non-radiative recombination at the interfaces, which is the primary loss mechanism for $V_{oc}$.

*   **Physics Role:** These layers dictate the 'Interface Recombination Velocity'. Optimal alignment between the perovskite VBM/CBM and the transport layer HOMO/LUMO is required for efficient charge extraction.
*   **ML Connection:** These are treated as **Categorical Sequence Descriptors**. CatBoost uses 'Symmetric Trees' to efficiently partition the high-dimensional space of 'Stack Architectures'. It identifies which sequences (e.g., Spiro-OMeTAD vs. PEDOT:PSS) offer better chemical protection against iodine migration while maintaining high $V_{oc}$ potential.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_white"

df_s = pd.read_parquet('../data/enriched_solar_panels.parquet')
print(f"Loaded {len(df_s)} experimental records.")

## 3. Data Distribution: Device Architecture Cardinality
Understanding the distribution of architectures and transport layers is crucial for assessing model coverage. We visualize the prevalence of n-i-p (conventional) vs p-i-n (inverted) structures and the dominant HTL materials. 

**Observation:** The dataset is heavily skewed towards the n-i-p architecture and Spiro-OMeTAD as an HTL, reflecting historical research trends. This imbalance suggests the ML model will be more robust in predicting n-i-p performance than p-i-n.

In [ ]:
fig1 = px.bar(df_s['cell_architecture'].value_counts().reset_index(), x='cell_architecture', y='count', 
             title="Dominance of n-i-p vs p-i-n Architectures", 
             labels={'cell_architecture': 'Cell Architecture', 'count': 'Sample Count'})
fig1.update_layout(showlegend=False, title_x=0.5)
fig1.show()

top_htls = df_s['htl_stack_sequence'].value_counts().head(10).reset_index()
fig2 = px.pie(top_htls, values='count', names='htl_stack_sequence', title="Top 10 HTL Materials in Research")
fig2.update_traces(textposition='inside', textinfo='percent+label')
fig2.show()

## 4. Stack Sequence Analysis: The HTL Influence
Hole Transport Layers (HTLs) are the 'weak link' in many perovskite stacks due to their hygroscopic nature or thermal instability. 

**Observation:** While Spiro-OMeTAD yields high median PCE, it often shows a wide variance in stability (not shown here, but inferred from stress tests). In contrast, inorganic HTLs like NiOx (common in p-i-n) show more consistent, albeit sometimes lower, peak efficiencies.

In [ ]:
top_htls_list = df_s['htl_stack_sequence'].value_counts().head(10).index
df_top_htl = df_s[df_s['htl_stack_sequence'].isin(top_htls_list)]

fig = px.box(
    df_top_htl, 
    x='htl_stack_sequence', 
    y='jv_default_pce', 
    color='cell_architecture', 
    title="Efficiency (PCE) Distribution by HTL Type",
    labels={'htl_stack_sequence': 'HTL Sequence', 'jv_default_pce': 'PCE (%)'}
)
fig.update_layout(width=1100, height=600, title_x=0.5, xaxis_tickangle=-45)
fig.show()

## 5. Stress Space Analysis: Testing Envelopes
The 'Stress Envelope' defines the operational limits explored by researchers. 

**Observation:** Most experiments cluster at room temperature (25°C) or moderate stress (65°C/50%RH). The sparsity of data at extreme conditions (85°C/85%RH) limits the ML model's ability to extrapolate degradation rates for long-term outdoor deployment.

In [ ]:
fig = px.scatter(
    df_s, 
    x='acc_temp', 
    y='acc_humidity', 
    size='jv_default_pce', 
    color='cell_architecture', 
    hover_data=['composition_long_form'],
    title="Experimental Stress Space: Temp vs. Humidity Coverage",
    labels={'acc_temp': 'Stress Temperature (°C)', 'acc_humidity': 'Relative Humidity (%RH)'}
)
fig.update_layout(width=900, height=600, title_x=0.5)
fig.show()

## 6. Physics-ML Interaction: Architecture vs HTL Performance
This heatmap visualizes the 'Synergy' between the transport layers and the device geometry. 

**Observation:** Certain HTLs are exclusively paired with specific architectures (e.g., Spiro-OMeTAD with n-i-p). The ML model must learn these conditional constraints to avoid predicting 'illegal' or physically impossible device stacks during inverse design.

In [ ]:
pivot_df = df_s[df_s['htl_stack_sequence'].isin(top_htls_list)].pivot_table(
    index='cell_architecture', columns='htl_stack_sequence', values='jv_default_pce', aggfunc='median'
)
fig = px.imshow(pivot_df, 
                labels=dict(x="HTL Sequence", y="Architecture", color="Median PCE (%)"),
                title="Architecture-HTL Synergy Heatmap",
                color_continuous_scale="Viridis")
fig.update_layout(title_x=0.5)
fig.show()

## 7. Performance-Stability Tradeoff: The Pareto Frontier
The Pareto frontier represents the ultimate engineering challenge: achieving the 'Top Right' quadrant where both PCE and Stability are maximized.

**Observation:** We see a clear 'inverse' trend (negative slope in the OLS line). Devices with extremely high PCE often suffer from lower $T_{80}$ lifetimes, highlighting the tradeoff between narrow-bandgap efficiency and chemical volatility. 

## 8. Limitations and Scientific Assumptions

### 8.1. Data Sparsity and Reporting Bias
*   **The 'Hero Cell' Bias:** Published literature tends to report 'best-in-class' results. The ML model is therefore trained on an optimistic subset of the true experimental space, likely overestimating average device performance.
*   **Missing Stress Vectors:** Factors like UV light intensity, oxygen concentration, and encapsulation quality are often poorly documented but are critical for degradation.

### 8.2. Physical Simplifications
*   **Constant Activation Energy:** The Arrhenius normalization assumes a single $E_a$ (0.6 eV) for all devices. In reality, $E_a$ varies significantly with the HTL choice and the presence of additives, which may introduce errors in the standardized lifetime $T_{80}$.
*   **Kinetic Resolution:** The analysis treats $T_{80}$ as a static value, ignoring the complex multi-phasic degradation curves (e.g., initial 'burn-in' vs. linear decay) that occur in real-world conditions.

In [ ]:
fig = px.scatter(
    df_s.dropna(subset=['stability_ts80m']), 
    x='jv_default_pce', 
    y='stability_ts80m', 
    color='cell_architecture', 
    log_y=True,
    hover_data=['composition_long_form'],
    trendline="ols",
    title="Efficiency-Stability Pareto Frontier",
    labels={'jv_default_pce': 'Initial PCE (%)', 'stability_ts80m': 'Lifetime T80 (Hours)'}
)
fig.update_layout(width=1000, height=600, title_x=0.5)
fig.show()